# Comprehensive Answers for GCC .NET Tech Interview (9+ Years Experience)

Since you have 9+ years of experience, I'll provide **senior-level answers** that demonstrate depth, practical experience, and architectural thinking.

---

## 1. .NET Core vs ASP.NET Core Difference

**Core Answer:**
- **.NET Core** is the **runtime/framework** (base class libraries, CLR, garbage collector, etc.)
- **ASP.NET Core** is the **web framework** built ON TOP of .NET Core (MVC, Web API, Razor Pages, SignalR, etc.)

**Key Distinctions:**

| Aspect | .NET Core | ASP.NET Core |
|--------|-----------|--------------|
| **What it is** | Cross-platform runtime + libraries | Web application framework |
| **Includes** | CLR, JIT, GC, BCL, file I/O, networking | Controllers, middleware, routing, DI |
| **Dependency** | Self-contained | Depends on .NET Core runtime |
| **Example** | Console app, services, libraries | Web APIs, MVC apps, SPA backends |

**Real-World Context:**
> "In my last project, we migrated from .NET Framework 4.8 to .NET 6. The runtime (.NET Core) gave us containerization benefits and performance improvements, while ASP.NET Core provided built-in DI, middleware pipeline, and minimal APIs that significantly reduced our codebase by ~30%."

**Key Takeaway for Senior:** 
> "Today, with .NET 5/6/7/8, the distinction is blurring as Microsoft unified the platform. Now it's just '.NET' with ASP.NET Core as the web framework."

---

## 2. OOPs Concepts

**Four Pillars with Real Examples:**

### 1. **Encapsulation** (Data Hiding)
```csharp
public class BankAccount
{
    private decimal _balance;  // Hidden
    private readonly List<Transaction> _transactions = new();
    
    public decimal GetBalance() => _balance;  // Controlled access
    
    public void Deposit(decimal amount)
    {
        if (amount <= 0) throw new ArgumentException("Amount must be positive");
        _balance += amount;
        _transactions.Add(new Transaction(amount, "Deposit"));
    }
}
```

### 2. **Inheritance** (IS-A Relationship)
```csharp
public abstract class PaymentProcessor
{
    public abstract void Process(decimal amount);
}

public class CreditCardProcessor : PaymentProcessor
{
    public override void Process(decimal amount) 
        => Console.WriteLine($"Processing {amount} via Credit Card");
}
```

### 3. **Polymorphism** (Many Forms)
```csharp
// Compile-time (Method Overloading)
public class Calculator
{
    public int Add(int a, int b) => a + b;
    public decimal Add(decimal a, decimal b) => a + b;
    public int Add(int a, int b, int c) => a + b + c;
}

// Runtime (Method Overriding)
public interface INotification
{
    void Send(string message);
}

public class EmailNotification : INotification 
{
    public void Send(string message) => Console.WriteLine($"Email: {message}");
}

public class SmsNotification : INotification
{
    public void Send(string message) => Console.WriteLine($"SMS: {message}");
}

// Usage
INotification notification = GetNotificationType();  // Runtime decision
notification.Send("Hello");  // Calls appropriate implementation
```

### 4. **Abstraction** (Hide Complexity)
```csharp
public interface IWeatherService  // Abstract contract
{
    Task<Weather> GetWeatherAsync(string city);
}

// Client only knows about IWeatherService, not implementation details
public class WeatherController : Controller
{
    private readonly IWeatherService _weatherService;
    
    public WeatherController(IWeatherService weatherService)
        => _weatherService = weatherService;
    
    public async Task<IActionResult> Get(string city)
        => Ok(await _weatherService.GetWeatherAsync(city));
}
```

**Senior Insight:**
> "In enterprise applications, we combine these: Abstraction through interfaces for contracts, Inheritance for base functionality, Encapsulation for domain integrity, and Polymorphism for extensibility."

---

## 3. Interface vs Class

| Aspect | Interface | Abstract Class | Concrete Class |
|--------|-----------|----------------|----------------|
| **Instantiation** | ❌ Cannot | ❌ Cannot | ✅ Can |
| **Implementation** | ❌ No (pre C# 8) | ✅ Partial | ✅ Full |
| **Fields** | ❌ No | ✅ Yes | ✅ Yes |
| **Constructors** | ❌ No | ✅ Yes | ✅ Yes |
| **Multiple Inheritance** | ✅ Multiple | ❌ Single | ❌ Single |
| **Access Modifiers** | Public only (implicit) | All | All |
| **Purpose** | Contract/Behavior | Base implementation | Complete implementation |

**Key Decisions (Senior-Level):**

- **Use Interface when:**
  - Defining a contract across unrelated classes
  - Need multiple inheritance (e.g., `IDisposable`, `IEnumerable`)
  - Mocking/Testing (DI containers work best with interfaces)
  - Plugin architectures

- **Use Abstract Class when:**
  - Sharing common code among related classes
  - Providing default implementations
  - Template Method pattern

- **Use Concrete Class when:**
  - You have complete implementation
  - No need for inheritance hierarchy

**Real-World:**
> "In our microservices architecture, we use interfaces extensively for contracts between services. For example, `IOrderService`, `IPaymentService` - allowing us to mock and test independently. Abstract classes we use for base repository patterns with common CRUD operations."

---

## 4. Struct vs Class

| Feature | Struct (Value Type) | Class (Reference Type) |
|---------|-------------------|----------------------|
| **Memory** | Stack (usually) | Heap |
| **Inheritance** | ❌ Cannot inherit | ✅ Can inherit |
| **Default Constructor** | ❌ No (parameterless not allowed) | ✅ Yes |
| **Nullability** | ❌ Cannot be null (unless nullable) | ✅ Can be null |
| **Performance** | Faster for small data | Heap allocation overhead |
| **Passing** | Pass by value (copy) | Pass by reference |
| **Garbage Collection** | ❌ Not collected | ✅ GC managed |
| **When to Use** | Small, immutable data | Large, complex objects |

**Practical Example:**
```csharp
// Struct - Used for small, immutable data
public struct Point
{
    public int X { get; }
    public int Y { get; }
    
    public Point(int x, int y) => (X, Y) = (x, y);
}

// Class - Used for complex behavior
public class Order
{
    public int Id { get; set; }
    public List<OrderItem> Items { get; set; } = new();
    public decimal TotalAmount => Items.Sum(i => i.Price * i.Quantity);
}
```

**Senior Insight:**
> "We choose structs for DTOs in high-performance scenarios. For example, in our real-time trading system, we use structs for price ticks (small, frequent data) and classes for orders (complex behavior). With .NET 8's `ref struct` and `Span<T>`, we've optimized even further."

---

## 5. Why Entity Framework over ADO.NET?

**Performance/Control vs Productivity:**

| Aspect | ADO.NET | Entity Framework |
|--------|---------|------------------|
| **Productivity** | ❌ Low (manual mapping) | ✅ High (auto mapping) |
| **Performance** | ✅ Excellent | ❌ Overhead (but improved with .NET 8) |
| **Control** | ✅ Full SQL control | ❌ Generated SQL (can be suboptimal) |
| **Maintenance** | ❌ High (schema changes) | ✅ Easy (migrations) |
| **Learning Curve** | ✅ Lower (just SQL) | ❌ Higher (LINQ, change tracking) |
| **Testability** | ❌ Hard (mock data) | ✅ Easy (in-memory/context) |
| **Change Tracking** | ❌ Manual | ✅ Automatic |
| **Lazy Loading** | ❌ No | ✅ Yes |

**When to choose EF (Senior Decision):**

✅ **Choose EF when:**
- Rapid development is priority
- Domain-driven design with complex relationships
- CRUD-heavy applications
- Need for migrations and version control
- You have an ORM-savvy team

❌ **Choose ADO.NET/Dapper when:**
- High performance is critical (microservices with millions of requests)
- Complex reports/stored procedures
- Bulk operations
- Legacy system integration

**Real-World Experience (Senior-Level):**
> "In my last project, we used a hybrid approach. For core domain logic, we used EF Core with repository pattern. For reporting and bulk operations, we used Dapper (micro-ORM). This gave us 80% productivity with EF and 95% performance with Dapper where needed."

**Example of Hybrid:**
```csharp
// EF Core for CRUD
public async Task<Order> GetOrderAsync(int id)
    => await _context.Orders
        .Include(o => o.Items)
        .FirstOrDefaultAsync(o => o.Id == id);

// Dapper for reports
public async Task<IEnumerable<SalesReport>> GetSalesReportAsync()
{
    using var connection = new SqlConnection(_connectionString);
    return await connection.QueryAsync<SalesReport>(
        @"SELECT Date, SUM(Amount) AS TotalSales 
          FROM Orders 
          GROUP BY Date");
}
```

**Performance Improvements in EF Core 8:**
- `ExecuteUpdate`/`ExecuteDelete` for bulk operations
- JSON column support
- Better LINQ translation
- Compiled queries

---

## 6. What if we register HttpClient as Singleton?

**Problem:**  
`HttpClient` was designed to be **reused**, but registering as Singleton **CAN BE DANGEROUS** if not configured properly.

### Issues with Singleton HttpClient:

❌ **DNS Stale Issue**  
When used as Singleton, DNS changes won't be respected because the underlying connection pool caches DNS.

❌ **Socket Exhaustion**  
If you create new HttpClient instances repeatedly (transient), you'll exhaust sockets.

✅ **Best Practice:**  
Use `IHttpClientFactory` with named/typed clients in .NET Core.

### Implementation:
```csharp
// ❌ Wrong - Singleton
services.AddSingleton<HttpClient>();  // Don't do this

// ✅ Correct - Using IHttpClientFactory
services.AddHttpClient("GitHub", client =>
{
    client.BaseAddress = new Uri("https://api.github.com/");
    client.DefaultRequestHeaders.UserAgent.ParseAdd("MyApp/1.0");
})
.AddPolicyHandler(GetRetryPolicy())  // Polly policies
.AddPolicyHandler(GetCircuitBreaker());

// ✅ Or Typed Client
services.AddHttpClient<IGitHubService, GitHubService>();

// Usage
public class GitHubService
{
    private readonly HttpClient _httpClient;
    
    public GitHubService(HttpClient httpClient)
    {
        _httpClient = httpClient;
    }
    
    public async Task<string> GetUserAsync(string username)
    {
        var response = await _httpClient.GetAsync($"users/{username}");
        response.EnsureSuccessStatusCode();
        return await response.Content.ReadAsStringAsync();
    }
}
```

**Senior Insight:**
> "In production, we always use `IHttpClientFactory` with Polly for resilience. For example, in our payment service, we have:
> - Retry (3 attempts with exponential backoff)
> - Circuit Breaker (fail fast after 5 failures)
> - Timeout (30 seconds)
> - Request/Response logging via DelegatingHandlers"

---

## 7. If we commit code in Main branch and find issues, Next Steps

### Immediate Action Plan:

**1. Stop the Pipeline** (if CI/CD in progress)
```yaml
# GitHub Actions - Manual approval required
jobs:
  deploy:
    environment: production
    runs-on: ubuntu-latest
    steps:
      - name: Deploy
        run: echo "Deploying..."
```

**2. Assess Impact (Blast Radius)**
- Critical? (Payment, Security) → **Rollback immediately**
- Minor? (UI, Documentation) → **Hotfix forward**

**3. Rollback Strategy**
```bash
# Option A: Revert commit
git revert <commit-hash>  # Creates new commit that undoes changes
git push origin main

# Option B: Reset (DANGEROUS - if nobody pulled)
git reset --hard HEAD~1
git push --force origin main

# Option C: Blue-Green deployment - just switch back
kubectl rollout undo deployment/myapp
```

**4. Communicate**
- Notify stakeholders immediately
- Document issue
- Update status in monitoring dashboards

**5. Root Cause Analysis** (After stabilization)
```markdown
## Post-Mortem Checklist
- What happened? (Timeline)
- Why did it happen? (Root cause)
- How did we detect? (Monitoring)
- How did we fix? (Actions)
- What will we change? (Prevention)
  - ✅ Pre-commit hooks
  - ✅ Better PR review process
  - ✅ Canary deployments
  - ✅ Feature flags
```

**6. Prevent Future Issues**
```csharp
// Example: Feature Flags
public class PaymentService
{
    private readonly IFeatureFlagService _featureFlags;
    
    public async Task<PaymentResult> ProcessPayment(PaymentRequest request)
    {
        if (_featureFlags.IsEnabled("NewPaymentFlow"))
        {
            return await NewFlowAsync(request);
        }
        return await OldFlowAsync(request);
    }
}
```

**Senior Insight (CI/CD Best Practices):**
> "At my current company, we follow:
> 1. **Protected Branches** - Require PR approvals (2 reviewers)
> 2. **CI Checks** - Build, Unit Tests, Integration Tests, Code Coverage
> 3. **Staging Deployment** - Automated QA testing
> 4. **Canary Deployment** - 5% → 20% → 50% → 100% traffic
> 5. **Rollback Automation** - Auto-rollback if error rate > 1%
> 6. **Feature Flags** - Kill switch without deployment
> 
> We learned this after a major incident where a database migration took down production for 15 minutes."

---

## 8. Load Balancer and Use Cases

### Types of Load Balancers:

| Type | Layer | Use Case | Tools |
|------|-------|----------|-------|
| **Layer 4** (Transport) | TCP/UDP | High performance, minimal processing | HAProxy, Nginx |
| **Layer 7** (Application) | HTTP/HTTPS | Smart routing, SSL offload | AWS ALB, Nginx, F5 |
| **Global** (DNS) | DNS | Geo-routing, Disaster Recovery | Cloudflare, Route 53 |

### Use Cases:

**1. Horizontal Scaling**
```
┌─────────┐
│   LB    │  (Round Robin, Least Connections, IP Hash)
└────┬────┘
     │
┌────┴────┐
│  App 1  │  (Instance 1)
├─────────┤
│  App 2  │  (Instance 2)
├─────────┤
│  App 3  │  (Instance 3)
└─────────┘
```

**2. SSL Termination**
```yaml
# Nginx Configuration
upstream backend {
    server app1:80;
    server app2:80;
}

server {
    listen 443 ssl;
    ssl_certificate /cert.pem;
    ssl_certificate_key /key.pem;
    
    location / {
        proxy_pass http://backend;
    }
}
```

**3. Health Checks**
```yaml
# Kubernetes
apiVersion: v1
kind: Service
metadata:
  name: myapp-service
spec:
  selector:
    app: myapp
  ports:
  - port: 80
    targetPort: 8080
  type: LoadBalancer
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: myapp
spec:
  replicas: 3
  selector:
    matchLabels:
      app: myapp
  template:
    metadata:
      labels:
        app: myapp
    spec:
      containers:
      - name: app
        image: myapp:latest
        ports:
        - containerPort: 8080
        livenessProbe:      # Kubernetes health check
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 30
          periodSeconds: 10
```

**Senior Insight:**
> "In our production setup:
> - **AWS ALB** for external traffic (Layer 7) with WAF
> - **Nginx** internal (Layer 4/7) for service mesh
> - **Global Traffic Management** via Route 53 (Active/Passive DR)
> 
> We implemented:
> - **Sticky Sessions** for stateful apps (using session affinity)
> - **Connection Draining** for graceful shutdowns
> - **Auto-scaling** based on CPU/Memory metrics
> - **Multi-AZ** deployment for high availability"

---

## 9. GitHub Actions and Flow

### GitHub Actions Flow:

```
┌─────────────┐
│ Push Code   │
└──────┬──────┘
       ▼
┌─────────────┐
│ Trigger     │  (on: push, pull_request, schedule)
└──────┬──────┘
       ▼
┌─────────────┐
│ Setup       │  (Checkout code, setup .NET)
└──────┬──────┘
       ▼
┌─────────────┐
│ Build       │  (Restore, Build, Test)
└──────┬──────┘
       ▼
┌─────────────┐
│ Test        │  (Unit, Integration, Code Coverage)
└──────┬──────┘
       ▼
┌─────────────┐
│ Package     │  (Create NuGet/Docker Image)
└──────┬──────┘
       ▼
┌─────────────┐
│ Release     │  (Publish to Artifacts/Container Registry)
└──────┬──────┘
       ▼
┌─────────────┐
│ Deploy      │  (Deploy to Environment)
└─────────────┘
```

### Complete CI/CD Pipeline Example:

```yaml
name: .NET 8 CI/CD Pipeline

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main ]
  schedule:
    - cron: '0 0 * * *'  # Daily security scan

jobs:
  build-and-test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        dotnet-version: ['8.0.x']
    
    steps:
    - uses: actions/checkout@v3
      with:
        fetch-depth: 0  # For versioning
    
    - name: Setup .NET
      uses: actions/setup-dotnet@v3
      with:
        dotnet-version: ${{ matrix.dotnet-version }}
    
    - name: Cache dependencies
      uses: actions/cache@v3
      with:
        path: ~/.nuget/packages
        key: ${{ runner.os }}-nuget-${{ hashFiles('**/*.csproj') }}
    
    - name: Restore dependencies
      run: dotnet restore
    
    - name: Build
      run: dotnet build --no-restore -c Release
    
    - name: Run Unit Tests
      run: dotnet test --no-build -c Release /p:CollectCoverage=true /p:CoverletOutputFormat=lcov
    
    - name: Upload coverage to Coveralls
      uses: coverallsapp/github-action@v2
    
    - name: Security Scan
      run: |
        dotnet tool install --global dotnet-retire
        dotnet-retire

  docker-build:
    needs: build-and-test
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    
    steps:
    - uses: actions/checkout@v3
    
    - name: Login to Docker Hub
      uses: docker/login-action@v2
      with:
        username: ${{ secrets.DOCKER_USERNAME }}
        password: ${{ secrets.DOCKER_PASSWORD }}
    
    - name: Build and push Docker image
      uses: docker/build-push-action@v4
      with:
        context: .
        push: true
        tags: |
          myapp:${{ github.sha }}
          myapp:latest
        cache-from: type=gha
        cache-to: type=gha,mode=max

  deploy-staging:
    needs: docker-build
    runs-on: ubuntu-latest
    environment: staging
    
    steps:
    - name: Deploy to Kubernetes
      run: |
        kubectl set image deployment/myapp myapp=myapp:${{ github.sha }}
        kubectl rollout status deployment/myapp

  deploy-production:
    needs: deploy-staging
    runs-on: ubuntu-latest
    environment: 
      name: production
      url: https://myapp.com
    if: github.ref == 'refs/heads/main'
    
    steps:
    - name: Manual Approval Required
      uses: actions/github-script@v6
      with:
        script: |
          const { data: reviews } = await github.rest.pulls.listReviews({
            owner: context.repo.owner,
            repo: context.repo.repo,
            pull_number: context.payload.pull_request.number
          });
          
          // Check for approvals
    
    - name: Deploy Production
      run: |
        echo "Deploying to production..."
        kubectl set image deployment/myapp myapp=myapp:${{ github.sha }}
        kubectl rollout status deployment/myapp
    
    - name: Smoke Test
      run: |
        curl -f https://myapp.com/health || exit 1
```

**Senior CI/CD Best Practices:**

1. **Branch Strategy**
   - `main` → Production (protected)
   - `develop` → Staging
   - `feature/*` → Development

2. **Security**
   - Store secrets in GitHub Secrets
   - Use OIDC for cloud authentication
   - Scan for vulnerabilities

3. **Performance**
   - Cache dependencies
   - Parallel jobs
   - Matrix testing (.NET versions, OS)

4. **Quality Gates**
   - Code coverage ≥ 80%
   - SonarCloud quality checks
   - Integration tests passing

5. **Observability**
   - Notifications (Slack/Teams)
   - Test reports
   - Deployment logs

---

## 10. Microservice Architecture: Benefits and Drawbacks

### Benefits (Senior View):

| Benefit | Explanation | Real Example |
|---------|-------------|--------------|
| **Independent Deployments** | Deploy one service without affecting others | "We deploy payment service 10x daily, while reporting service weekly" |
| **Technology Diversity** | Use best tool for the job | "Orders use .NET, Analytics use Python, Recommendations use Go" |
| **Team Autonomy** | Teams own their services | "2-pizza teams: 6 developers per service" |
| **Scalability** | Scale individual services | "Checkout service scaled 50x during Black Friday, other services unchanged" |
| **Resilience** | Isolate failures | "Payment failure doesn't crash product catalog" |
| **Faster Time-to-Market** | Parallel development | "3 teams simultaneously working on different features" |

### Drawbacks (Senior View):

| Drawback | Impact | Mitigation |
|----------|--------|------------|
| **Distributed Complexity** | Network latency, failures | Circuit breakers, retries, timeouts |
| **Data Consistency** | No ACID transactions | Saga pattern, eventual consistency |
| **Operational Overhead** | Monitoring 100+ services | Centralized logging (ELK), distributed tracing (Jaeger) |
| **Debugging** | Complex to trace | Correlation IDs, APM tools (Datadog, NewRelic) |
| **Testing** | Integration complexity | Contract testing (Pact), test containers |
| **Team Coordination** | Communication overhead | API contracts, OpenAPI/Swagger |

### Architectural Example:

```
┌─────────────────────────────────────────────────────────┐
│                    API Gateway (Ocelot)                 │
│      - Authentication, Rate Limiting, Routing           │
└─────────────────────────────────────────────────────────┘
                         │
        ┌────────────────┼────────────────┐
        ▼                ▼                ▼
┌──────────────┐ ┌──────────────┐ ┌──────────────┐
│  Order       │ │  Payment     │ │  Inventory   │
│  Service     │◄┤  Service     │◄┤  Service     │
│  .NET 8      │ │  .NET 8      │ │  .NET 8      │
└──────┬───────┘ └──────┬───────┘ └──────┬───────┘
       │                │                │
       ▼                ▼                ▼
┌──────────────┐ ┌──────────────┐ ┌──────────────┐
│  SQL Server  │ │  PostgreSQL  │ │  Redis       │
│  (Orders)    │ │  (Payments)  │ │  (Cache)     │
└──────────────┘ └──────────────┘ └──────────────┘
       │                │                │
       └────────────────┼────────────────┘
                        ▼
              ┌─────────────────────┐
              │  Message Bus        │
              │  (RabbitMQ/Kafka)   │
              └─────────────────────┘
                        │
        ┌───────────────┼───────────────┐
        ▼               ▼               ▼
┌──────────────┐ ┌──────────────┐ ┌──────────────┐
│  Shipping    │ │  Notification│ │  Analytics   │
│  Service     │ │  Service     │ │  Service     │
└──────────────┘ └──────────────┘ └──────────────┘
```

### Real-World Patterns (Senior-Level):

**1. Saga Pattern (Data Consistency)**
```csharp
public class OrderSaga
{
    public async Task ProcessOrder(Order order)
    {
        try
        {
            await ReserveInventory(order);     // Step 1
            await ProcessPayment(order);       // Step 2
            await CreateShipment(order);       // Step 3
            await SendNotification(order);     // Step 4
        }
        catch (Exception ex)
        {
            // Compensating transactions
            await ReversePayment(order);       // Rollback Step 2
            await ReleaseInventory(order);     // Rollback Step 1
            throw;
        }
    }
}
```

**2. CQRS (Command Query Responsibility Segregation)**
```csharp
// Write side (Commands)
public class CreateOrderCommand
{
    public int ProductId { get; set; }
    public int Quantity { get; set; }
}

// Read side (Queries)
public class GetOrderDetailsQuery
{
    public int OrderId { get; set; }
}

// Separate models
public class OrderWriteModel { /* For writes */ }
public class OrderReadModel { /* For reads, denormalized */ }
```

**3. Circuit Breaker Pattern**
```csharp
// Using Polly
services.AddHttpClient<IPaymentService, PaymentService>()
    .AddPolicyHandler(Policy
        .Handle<HttpRequestException>()
        .CircuitBreakerAsync(
            handledEventsAllowedBeforeBreaking: 3,
            durationOfBreak: TimeSpan.FromSeconds(30),
            onBreak: (ex, duration) => {
                // Alert, fallback to cache
            },
            onReset: () => {
                // Service restored
            }
        ));
```

### When NOT to use Microservices:

- Small teams (< 10 developers)
- Simple CRUD applications
- Tightly coupled business processes
- Limited DevOps capabilities
- Budget constraints

**Senior Verdict:**
> "In my current project, we use a modular monolith with clear bounded contexts. We extract services to microservices when:
> - Team size grows (12+ developers)
> - Need for independent scaling
> - Different technology requirements
> 
> This is our 'Strangler Fig' pattern - gradually extracting services as needed."

---

## 11. Exception Handling

### Best Practices for Senior Developers:

**1. Use Specific Exceptions**
```csharp
// ❌ Bad
throw new Exception("Something went wrong");

// ✅ Good
throw new ValidationException("Order total exceeds limit");
throw new PaymentGatewayException("Payment failed: insufficient funds");
throw new ConcurrencyException("Record was modified by another user");
```

**2. Global Exception Handling**
```csharp
public class GlobalExceptionHandler : IExceptionHandler
{
    private readonly ILogger<GlobalExceptionHandler> _logger;
    
    public GlobalExceptionHandler(ILogger<GlobalExceptionHandler> logger)
        => _logger = logger;
    
    public async ValueTask<bool> TryHandleAsync(
        HttpContext context, 
        Exception exception, 
        CancellationToken cancellationToken)
    {
        _logger.LogError(exception, "Unhandled exception occurred");
        
        var response = new ErrorResponse
        {
            RequestId = context.TraceIdentifier,
            Timestamp = DateTime.UtcNow,
            Message = "An error occurred processing your request"
        };
        
        context.Response.StatusCode = exception switch
        {
            ValidationException => StatusCodes.Status400BadRequest,
            NotFoundException => StatusCodes.Status404NotFound,
            UnauthorizedException => StatusCodes.Status401Unauthorized,
            _ => StatusCodes.Status500InternalServerError
        };
        
        await context.Response.WriteAsJsonAsync(response);
        return true;
    }
}
```

**3. Application-Level Exception Handling**
```csharp
public class OrderService
{
    private readonly ILogger _logger;
    
    public async Task<OrderResult> CreateOrder(OrderRequest request)
    {
        try
        {
            // Business logic
            ValidateOrder(request);
            
            // Database operation
            await SaveOrderAsync(request);
            
            // External API call
            await ProcessPaymentAsync(request);
            
            return new SuccessResult { OrderId = request.Id };
        }
        catch (ValidationException ex)
        {
            _logger.LogWarning(ex, "Validation failed: {Errors}", ex.Errors);
            throw; // Let global handler manage
        }
        catch (PaymentException ex)
        {
            _logger.LogError(ex, "Payment processing failed for order {OrderId}", request.Id);
            // Retry or fallback
            throw new OrderProcessingException("Payment failed", ex);
        }
        catch (DbUpdateConcurrencyException ex)
        {
            _logger.LogError(ex, "Concurrency conflict for order {OrderId}", request.Id);
            // Retry with entity refresh
            throw;
        }
        catch (Exception ex)
        {
            _logger.LogCritical(ex, "Unexpected error processing order {OrderId}", request.Id);
            throw new OrderProcessingException("Unexpected error", ex);
        }
    }
}
```

**4. Async Exception Handling**
```csharp
// Fire-and-forget with error handling
public async Task ProcessBackgroundJobAsync()
{
    try
    {
        await Task.Run(async () => 
        {
            try
            {
                await DoWorkAsync();
            }
            catch (Exception ex)
            {
                _logger.LogError(ex, "Background job failed");
                // Notify via monitoring
                await _alertingService.AlertAsync("Background job failed", ex);
            }
        });
    }
    catch (Exception ex)
    {
        // Catch unobserved exceptions
        _logger.LogCritical(ex, "Critical background job failure");
    }
}
```

**5. Exception Handling in ASP.NET Core Middleware**
```csharp
app.UseExceptionHandler(appBuilder =>
{
    appBuilder.Run(async context =>
    {
        var exception = context.Features
            .Get<IExceptionHandlerFeature>()?
            .Error;
            
        if (exception != null)
        {
            context.Response.StatusCode = 500;
            await context.Response.WriteAsync("An error occurred");
        }
    });
});

// Developer Exception Page (Development only)
if (app.Environment.IsDevelopment())
{
    app.UseDeveloperExceptionPage();
}
```

### Exception Handling Checklist (Senior):

- [ ] **Log All Exceptions** - Use structured logging (Serilog)
- [ ] **No `catch (Exception)` without logging** - Always log
- [ ] **Use `finally` or `using` for resource cleanup** - Dispose
- [ ] **Avoid exception swallowing** - Don't catch and ignore
- [ ] **Return meaningful messages to clients** - Not stack traces
- [ ] **Use `Exception.Data`** - Add contextual information
- [ ] **Implement retry logic** - For transient failures
- [ ] **Use Polly for resilience** - Retry, circuit breaker, timeout
- [ ] **Monitor exceptions** - Application Insights, Sentry

**Senior Insight:**
> "We have a comprehensive exception handling strategy:
> 1. **Domain Exceptions** - For business rules (e.g., `InsufficientFundsException`)
> 2. **Technical Exceptions** - For infrastructure (e.g., `DatabaseException`)
> 3. **Global Handler** - Catches unhandled exceptions
> 4. **Health Checks** - Monitor error rates
> 5. **Alerts** - PagerDuty for critical errors
> 
> We use **structured logging** with Serilog to capture:
> - `{CorrelationId}` - To trace across services
> - `{UserId}` - To audit
> - `{Exception}` - Full exception details
> - `{Request}` - Request details for debugging"

---

## 12. Caching Strategies

### Types of Caching:

| Type | Location | Use Case | Tools |
|------|----------|----------|-------|
| **In-Memory** | Application | Frequently accessed data | IMemoryCache, ConcurrentDictionary |
| **Distributed** | Shared Cache | Multi-instance sharing | Redis, Azure Cache, NCache |
| **CDN** | Edge | Static assets | CloudFront, Cloudflare |
| **Database** | Query Cache | Expensive queries | SQL Server Query Store |
| **Client** | Browser | Static resources | Cache-Control headers |

### 1. In-Memory Caching (IMemoryCache)
```csharp
public class ProductService
{
    private readonly IMemoryCache _cache;
    private readonly ILogger<ProductService> _logger;
    
    public ProductService(IMemoryCache cache, ILogger<ProductService> logger)
        => (_cache, _logger) = (cache, logger);
    
    public async Task<Product> GetProductAsync(int id)
    {
        var cacheKey = $"Product_{id}";
        
        if (_cache.TryGetValue(cacheKey, out Product cachedProduct))
        {
            _logger.LogInformation("Cache hit for Product {ProductId}", id);
            return cachedProduct;
        }
        
        _logger.LogInformation("Cache miss for Product {ProductId}", id);
        var product = await GetFromDatabaseAsync(id);
        
        var cacheOptions = new MemoryCacheEntryOptions
        {
            AbsoluteExpirationRelativeToNow = TimeSpan.FromMinutes(10),
            SlidingExpiration = TimeSpan.FromMinutes(2),
            Priority = CacheItemPriority.High,
            Size = 1024
        };
        
        // Optional: Background refresh
        cacheOptions.RegisterPostEvictionCallback((key, value, reason, state) =>
        {
            _logger.LogInformation("Cache evicted: {Key}, Reason: {Reason}", key, reason);
            // Refresh cache in background
            Task.Run(() => RefreshCacheAsync(id));
        });
        
        _cache.Set(cacheKey, product, cacheOptions);
        return product;
    }
}
```

### 2. Distributed Caching (Redis)
```csharp
public class RedisCacheService
{
    private readonly IDistributedCache _cache;
    private readonly ILogger _logger;
    
    public RedisCacheService(IDistributedCache cache, ILogger logger)
        => (_cache, _logger) = (cache, logger);
    
    public async Task<T> GetOrSetAsync<T>(string key, Func<Task<T>> factory, TimeSpan? expiry = null)
    {
        var cached = await _cache.GetStringAsync(key);
        
        if (!string.IsNullOrEmpty(cached))
        {
            _logger.LogDebug("Cache hit for {Key}", key);
            return JsonSerializer.Deserialize<T>(cached);
        }
        
        _logger.LogDebug("Cache miss for {Key}", key);
        var data = await factory();
        
        if (data != null)
        {
            var options = new DistributedCacheEntryOptions();
            options.SetAbsoluteExpiration(expiry ?? TimeSpan.FromMinutes(5));
            
            var json = JsonSerializer.Serialize(data);
            await _cache.SetStringAsync(key, json, options);
        }
        
        return data;
    }
    
    public async Task InvalidateAsync(string key)
    {
        await _cache.RemoveAsync(key);
        _logger.LogInformation("Cache invalidated: {Key}", key);
    }
}
```

### 3. Cache-Aside Pattern
```csharp
public class OrderService
{
    private readonly IOrderRepository _repository;
    private readonly IDistributedCache _cache;
    
    public async Task<Order> GetOrderAsync(int id)
    {
        var cacheKey = $"Order:{id}";
        
        // 1. Try cache
        var cached = await _cache.GetStringAsync(cacheKey);
        if (cached != null)
            return JsonSerializer.Deserialize<Order>(cached);
        
        // 2. Query database
        var order = await _repository.GetByIdAsync(id);
        
        // 3. Store in cache
        if (order != null)
        {
            var options = new DistributedCacheEntryOptions
            {
                AbsoluteExpiration = DateTimeOffset.Now.AddMinutes(10)
            };
            
            await _cache.SetStringAsync(
                cacheKey,
                JsonSerializer.Serialize(order),
                options
            );
        }
        
        return order;
    }
}
```

### 4. Cache Invalidation Strategies

**Time-Based:**
```csharp
// Absolute
cacheOptions.AbsoluteExpirationRelativeToNow = TimeSpan.FromMinutes(5);

// Sliding (extend on access)
cacheOptions.SlidingExpiration = TimeSpan.FromMinutes(2);

// No expiration (manual invalidation)
cacheOptions.SetPriority(CacheItemPriority.NeverRemove);
```

**Event-Based Invalidation:**
```csharp
public class ProductCacheInvalidator
{
    private readonly IDistributedCache _cache;
    private readonly IEventBus _eventBus;
    
    public void Setup()
    {
        // Invalidate when product updates
        _eventBus.Subscribe<ProductUpdatedEvent>(async evt =>
        {
            await _cache.RemoveAsync($"Product:{evt.ProductId}");
            await _cache.RemoveAsync($"Products:Category:{evt.CategoryId}");
            await _cache.RemoveAsync($"Products:All");
        });
        
        // Invalidate when category changes
        _eventBus.Subscribe<CategoryChangedEvent>(async evt =>
        {
            await _cache.RemoveAsync($"Category:{evt.CategoryId}");
            await _cache.RemoveAsync($"Products:Category:{evt.CategoryId}");
        });
    }
}
```

### 5. Advanced Caching Patterns

**Cache-Through:**
```csharp
// Reads through cache - cache loads data if missing
public class CacheThroughRepository<T>
{
    public async Task<T> GetAsync(string key)
    {
        var value = await _cache.GetAsync(key);
        if (value == null)
        {
            value = await _database.GetAsync(key);
            await _cache.SetAsync(key, value);
        }
        return value;
    }
}
```

**Write-Through:**
```csharp
// Writes to both cache and database
public async Task UpdateAsync(T entity)
{
    await _database.UpdateAsync(entity);
    await _cache.SetAsync(entity.Id, entity);
}
```

**Write-Behind:**
```csharp
// Writes to cache, async writes to database
public class WriteBehindCache
{
    private readonly ConcurrentQueue<Entity> _writeQueue = new();
    private readonly Timer _flushTimer;
    
    public async Task UpdateAsync(Entity entity)
    {
        _writeQueue.Enqueue(entity);
        await _cache.SetAsync(entity.Id, entity);
    }
    
    private async Task FlushQueueAsync()
    {
        while (_writeQueue.TryDequeue(out var entity))
        {
            await _database.UpdateAsync(entity);
        }
    }
}
```

### Caching Best Practices:

1. **Cache Only What's Expensive**
   - ✅ Database queries
   - ✅ API calls
   - ✅ Computations
   - ❌ Simple calculations

2. **Use Appropriate Expiration**
   - Static data: Long TTL
   - Dynamic data: Short TTL or event-driven invalidation

3. **Monitor Cache Performance**
   - Hit ratio
   - Memory usage
   - Eviction rates

4. **Handle Stale Data**
   ```csharp
   // Use stale data while refreshing
   if (_cache.TryGetValue(key, out var cached))
   {
       // Return stale data but refresh in background
       Task.Run(() => RefreshCacheAsync(key));
       return cached;
   }
   ```

5. **Cache Patterns in Microservices**
   - **Local Cache**: For each service instance
   - **Shared Cache**: Redis for cross-service sharing
   - **Multi-Level Cache**: Local L1 + Distributed L2

**Senior Insight:**
> "In production, we implemented a multi-layered caching strategy:
> 
> **Layer 1 (L1) - In-Memory (IMemoryCache)**
> - 5-minute expiration
> - 10,000 items max
> - Used for frequently accessed products
> 
> **Layer 2 (L2) - Redis Distributed Cache**
> - 1-hour expiration
> - All products
> - Cross-instance sharing
> 
> **Layer 3 - CDN for static assets**
> - CloudFront with custom origin
> - Cache-Control: max-age=86400
> 
> We also use **Cache Stampede Protection**:
> ```csharp
> private static readonly SemaphoreSlim _cacheLock = new(1, 1);
> 
> public async Task<T> GetWithLockAsync<T>(string key, Func<Task<T>> factory)
> {
>     if (_cache.TryGetValue(key, out T cached))
>         return cached;
>     
>     await _cacheLock.WaitAsync();
>     try
>     {
>         // Double-check after acquiring lock
>         if (_cache.TryGetValue(key, out cached))
>             return cached;
>         
>         var data = await factory();
>         _cache.Set(key, data);
>         return data;
>     }
>     finally
>     {
>         _cacheLock.Release();
>     }
> }
> ```"

---

## 13. Design Patterns and Types

### Complete Classification with Real Examples:

## 🏗️ **Creational Patterns** (Object Creation)

### 1. Singleton
```csharp
// Thread-safe, lazy initialization
public sealed class Configuration
{
    private static readonly Lazy<Configuration> _instance = 
        new(() => new Configuration());
    
    private Configuration() { }
    
    public static Configuration Instance => _instance.Value;
    
    public Dictionary<string, string> Settings { get; } = new();
}
```
**Use Case:** Application configuration, logging, database connection pool

### 2. Factory Method
```csharp
public interface ILogger
{
    void Log(string message);
}

public class LoggerFactory
{
    public ILogger CreateLogger(string type)
    {
        return type.ToLower() switch
        {
            "file" => new FileLogger(),
            "database" => new DatabaseLogger(),
            "console" => new ConsoleLogger(),
            _ => throw new ArgumentException($"Unknown logger type: {type}")
        };
    }
}
```
**Use Case:** Creating objects without specifying concrete classes

### 3. Abstract Factory
```csharp
public interface IUIElement
{
    void Render();
}

public interface IUIFactory
{
    IButton CreateButton();
    ITextBox CreateTextBox();
    IWindow CreateWindow();
}

public class WindowsUIFactory : IUIFactory { /* Windows implementations */ }
public class MacUIFactory : IUIFactory { /* Mac implementations */ }
```
**Use Case:** Creating families of related objects

### 4. Builder
```csharp
public class DatabaseConnectionBuilder
{
    private string _host;
    private int _port;
    private string _database;
    private string _username;
    private string _password;
    private bool _sslEnabled;
    private int _timeout = 30;
    
    public DatabaseConnectionBuilder SetHost(string host)
    {
        _host = host;
        return this;
    }
    
    public DatabaseConnectionBuilder SetPort(int port)
    {
        _port = port;
        return this;
    }
    
    public DatabaseConnectionBuilder SetDatabase(string database)
    {
        _database = database;
        return this;
    }
    
    public DatabaseConnectionBuilder WithCredentials(string username, string password)
    {
        _username = username;
        _password = password;
        return this;
    }
    
    public DatabaseConnectionBuilder WithSSL(bool enabled)
    {
        _sslEnabled = enabled;
        return this;
    }
    
    public DatabaseConnection Build()
    {
        // Validation
        if (string.IsNullOrEmpty(_host))
            throw new InvalidOperationException("Host is required");
        
        return new DatabaseConnection
        {
            Host = _host,
            Port = _port,
            Database = _database,
            Username = _username,
            Password = _password,
            SSLEnabled = _sslEnabled,
            Timeout = _timeout
        };
    }
}

// Usage
var connection = new DatabaseConnectionBuilder()
    .SetHost("localhost")
    .SetPort(5432)
    .SetDatabase("mydb")
    .WithCredentials("admin", "password")
    .WithSSL(true)
    .Build();
```
**Use Case:** Complex object construction (e.g., DbContext options, HttpClient configuration)

### 5. Prototype
```csharp
public interface IPrototype<T>
{
    T Clone();
}

public class User : IPrototype<User>
{
    public int Id { get; set; }
    public string Name { get; set; }
    public Address Address { get; set; }
    
    public User Clone()
    {
        // Deep copy
        return new User
        {
            Id = this.Id,
            Name = this.Name,
            Address = new Address
            {
                Street = this.Address.Street,
                City = this.Address.City
            }
        };
    }
}
```
**Use Case:** Creating copies of objects (shallow/deep)

## 🏛️ **Structural Patterns** (Object Structure)

### 6. Adapter
```csharp
// Third-party library
public class LegacyPaymentGateway
{
    public bool ProcessPayment(decimal amount, string cardNumber, string expiry)
    {
        // Complex legacy code
        return true;
    }
}

// Modern interface
public interface IPaymentProcessor
{
    Task<PaymentResult> ProcessAsync(PaymentRequest request);
}

// Adapter
public class PaymentAdapter : IPaymentProcessor
{
    private readonly LegacyPaymentGateway _legacy;
    
    public PaymentAdapter(LegacyPaymentGateway legacy)
        => _legacy = legacy;
    
    public async Task<PaymentResult> ProcessAsync(PaymentRequest request)
    {
        // Adapt modern request to legacy
        var success = _legacy.ProcessPayment(
            request.Amount,
            request.Card.Number,
            $"{request.Card.ExpiryMonth}/{request.Card.ExpiryYear}"
        );
        
        return new PaymentResult
        {
            Success = success,
            TransactionId = success ? Guid.NewGuid().ToString() : null
        };
    }
}
```
**Use Case:** Integrating with legacy systems or third-party libraries

### 7. Decorator
```csharp
public interface IDataSource
{
    string Read();
    void Write(string data);
}

public class FileDataSource : IDataSource
{
    public string Read() => File.ReadAllText("data.txt");
    public void Write(string data) => File.WriteAllText("data.txt", data);
}

public abstract class DataSourceDecorator : IDataSource
{
    protected IDataSource _wrapped;
    
    public DataSourceDecorator(IDataSource source) => _wrapped = source;
    
    public virtual string Read() => _wrapped.Read();
    public virtual void Write(string data) => _wrapped.Write(data);
}

public class CompressionDecorator : DataSourceDecorator
{
    public CompressionDecorator(IDataSource source) : base(source) { }
    
    public override string Read()
    {
        var compressed = base.Read();
        return Decompress(compressed);
    }
    
    public override void Write(string data)
    {
        var compressed = Compress(data);
        base.Write(compressed);
    }
}

public class EncryptionDecorator : DataSourceDecorator
{
    public EncryptionDecorator(IDataSource source) : base(source) { }
    
    public override string Read()
    {
        var encrypted = base.Read();
        return Decrypt(encrypted);
    }
    
    public override void Write(string data)
    {
        var encrypted = Encrypt(data);
        base.Write(encrypted);
    }
}

// Usage
var source = new FileDataSource();
var compressed = new CompressionDecorator(source);
var encrypted = new EncryptionDecorator(compressed);
encrypted.Write("Sensitive data"); // Encrypted + Compressed
```
**Use Case:** Adding responsibilities dynamically (logging, caching, validation)

### 8. Facade
```csharp
public class OrderFacade
{
    private readonly IInventoryService _inventory;
    private readonly IPaymentService _payment;
    private readonly IShippingService _shipping;
    private readonly INotificationService _notification;
    
    public OrderFacade(
        IInventoryService inventory,
        IPaymentService payment,
        IShippingService shipping,
        INotificationService notification)
    {
        _inventory = inventory;
        _payment = payment;
        _shipping = shipping;
        _notification = notification;
    }
    
    public async Task<OrderResult> PlaceOrderAsync(OrderRequest request)
    {
        // 1. Check inventory
        if (!await _inventory.CheckAvailabilityAsync(request.ProductId, request.Quantity))
            return new OrderResult { Success = false, Error = "Product unavailable" };
        
        // 2. Reserve inventory
        await _inventory.ReserveAsync(request.ProductId, request.Quantity);
        
        // 3. Process payment
        var paymentResult = await _payment.ProcessAsync(request.Payment);
        if (!paymentResult.Success)
        {
            await _inventory.ReleaseAsync(request.ProductId, request.Quantity);
            return new OrderResult { Success = false, Error = "Payment failed" };
        }
        
        // 4. Create shipment
        var shipment = await _shipping.CreateShipmentAsync(request);
        
        // 5. Send notification
        await _notification.SendOrderConfirmationAsync(request.CustomerId, shipment);
        
        return new OrderResult { Success = true, OrderId = request.OrderId };
    }
}
```
**Use Case:** Simplifying complex subsystems for clients

### 9. Proxy
```csharp
public interface IImage
{
    void Display();
}

public class RealImage : IImage
{
    private readonly string _filename;
    
    public RealImage(string filename)
    {
        _filename = filename;
        LoadFromDisk();
    }
    
    private void LoadFromDisk()
        => Console.WriteLine($"Loading image: {_filename}");
    
    public void Display()
        => Console.WriteLine($"Displaying image: {_filename}");
}

public class ProxyImage : IImage
{
    private readonly string _filename;
    private RealImage _realImage;
    
    public ProxyImage(string filename) => _filename = filename;
    
    public void Display()
    {
        if (_realImage == null)
        {
            _realImage = new RealImage(_filename); // Lazy loading
        }
        _realImage.Display();
    }
}

// Usage
IImage image = new ProxyImage("photo.jpg");
image.Display(); // Loads from disk
image.Display(); // Uses cached instance
```
**Use Case:** Lazy loading, access control, logging, caching

## 🔄 **Behavioral Patterns** (Interaction)

### 10. Observer
```csharp
public class Stock
{
    private readonly List<IInvestor> _investors = new();
    private decimal _price;
    
    public string Symbol { get; }
    public decimal Price
    {
        get => _price;
        set
        {
            if (_price != value)
            {
                _price = value;
                NotifyInvestors();
            }
        }
    }
    
    public Stock(string symbol, decimal price)
    {
        Symbol = symbol;
        _price = price;
    }
    
    public void Attach(IInvestor investor) => _investors.Add(investor);
    public void Detach(IInvestor investor) => _investors.Remove(investor);
    
    private void NotifyInvestors()
    {
        foreach (var investor in _investors)
            investor.Update(this);
    }
}

public interface IInvestor
{
    void Update(Stock stock);
}

public class Investor : IInvestor
{
    public string Name { get; }
    
    public Investor(string name) => Name = name;
    
    public void Update(Stock stock)
        => Console.WriteLine($"{Name} notified: {stock.Symbol} changed to ${stock.Price}");
}

// Usage
Stock apple = new("AAPL", 150.00m);
Investor john = new("John");
Investor jane = new("Jane");

apple.Attach(john);
apple.Attach(jane);

apple.Price = 155.50m; // Both investors notified
```
**Use Case:** Event handling, pub/sub systems, data binding

### 11. Strategy
```csharp
public interface ITaxCalculator
{
    decimal Calculate(decimal amount);
}

public class USTaxCalculator : ITaxCalculator
{
    private const decimal TaxRate = 0.07m;
    public decimal Calculate(decimal amount) => amount * TaxRate;
}

public class EuropeTaxCalculator : ITaxCalculator
{
    private const decimal TaxRate = 0.20m;
    public decimal Calculate(decimal amount) => amount * TaxRate;
}

public class NoTaxCalculator : ITaxCalculator
{
    public decimal Calculate(decimal amount) => 0;
}

public class OrderProcessor
{
    private ITaxCalculator _taxCalculator;
    
    public OrderProcessor(ITaxCalculator taxCalculator)
        => _taxCalculator = taxCalculator;
    
    public void SetTaxCalculator(ITaxCalculator taxCalculator)
        => _taxCalculator = taxCalculator;
    
    public Invoice ProcessOrder(Order order)
    {
        var subtotal = order.Items.Sum(i => i.Price * i.Quantity);
        var tax = _taxCalculator.Calculate(subtotal);
        
        return new Invoice
        {
            Subtotal = subtotal,
            Tax = tax,
            Total = subtotal + tax
        };
    }
}
```
**Use Case:** Algorithms that vary independently from clients

### 12. Command
```csharp
public interface ICommand
{
    void Execute();
    void Undo();
}

public class CalculatorCommand : ICommand
{
    private readonly Calculator _calculator;
    private readonly char _operator;
    private readonly int _operand;
    private int _previousValue;
    
    public CalculatorCommand(Calculator calculator, char operator_, int operand)
    {
        _calculator = calculator;
        _operator = operator_;
        _operand = operand;
    }
    
    public void Execute()
    {
        _previousValue = _calculator.CurrentValue;
        _calculator.Operation(_operator, _operand);
    }
    
    public void Undo()
    {
        _calculator.CurrentValue = _previousValue;
    }
}

public class Calculator
{
    public int CurrentValue { get; set; } = 0;
    
    public void Operation(char operator_, int operand)
    {
        CurrentValue = operator_ switch
        {
            '+' => CurrentValue + operand,
            '-' => CurrentValue - operand,
            '*' => CurrentValue * operand,
            '/' => CurrentValue / operand,
            _ => CurrentValue
        };
        Console.WriteLine($"Current value: {CurrentValue}");
    }
}
```
**Use Case:** Undo/redo, transaction, job queues

### 13. Chain of Responsibility
```csharp
public interface IRequestHandler
{
    void HandleRequest(Request request);
    IRequestHandler SetNext(IRequestHandler next);
}

public abstract class Handler : IRequestHandler
{
    protected IRequestHandler _next;
    
    public IRequestHandler SetNext(IRequestHandler next)
    {
        _next = next;
        return next;
    }
    
    public abstract void HandleRequest(Request request);
}

public class ValidationHandler : Handler
{
    public override void HandleRequest(Request request)
    {
        if (string.IsNullOrEmpty(request.Data))
        {
            Console.WriteLine("Validation failed: Empty request");
            return;
        }
        Console.WriteLine("Validation passed");
        _next?.HandleRequest(request);
    }
}

public class AuthenticationHandler : Handler
{
    public override void HandleRequest(Request request)
    {
        if (request.UserRole != "Admin")
        {
            Console.WriteLine("Authentication failed: Insufficient permissions");
            return;
        }
        Console.WriteLine("Authentication passed");
        _next?.HandleRequest(request);
    }
}

public class LoggingHandler : Handler
{
    public override void HandleRequest(Request request)
    {
        Console.WriteLine($"Processing request: {request.UserRole} - {request.Data}");
        _next?.HandleRequest(request);
    }
}
```
**Use Case:** Processing pipelines, middleware, logging, validation

### 14. Template Method
```csharp
public abstract class DataProcessor
{
    // Template method
    public void Process()
    {
        LoadData();
        ProcessData();
        SaveData();
        Cleanup();
    }
    
    protected abstract void LoadData();
    protected abstract void ProcessData();
    protected virtual void SaveData() => Console.WriteLine("Saving data...");
    protected virtual void Cleanup() => Console.WriteLine("Cleaning up...");
}

public class CsvProcessor : DataProcessor
{
    protected override void LoadData()
        => Console.WriteLine("Loading CSV file...");
    
    protected override void ProcessData()
        => Console.WriteLine("Processing CSV data...");
}

public class XmlProcessor : DataProcessor
{
    protected override void LoadData()
        => Console.WriteLine("Loading XML file...");
    
    protected override void ProcessData()
        => Console.WriteLine("Processing XML data...");
    
    protected override void Cleanup()
    {
        Console.WriteLine("Validating XML schema...");
        base.Cleanup();
    }
}
```
**Use Case:** Frameworks, base classes with customizable steps

### 15. State
```csharp
public interface IOrderState
{
    void Process(Order order);
    void Ship(Order order);
    void Cancel(Order order);
}

public class NewOrderState : IOrderState
{
    public void Process(Order order)
    {
        Console.WriteLine("Processing order...");
        order.SetState(new ProcessingOrderState());
    }
    
    public void Ship(Order order)
        => Console.WriteLine("Cannot ship: Order not processed yet");
    
    public void Cancel(Order order)
    {
        Console.WriteLine("Cancelling order...");
        order.SetState(new CancelledOrderState());
    }
}

public class ProcessingOrderState : IOrderState
{
    public void Process(Order order)
        => Console.WriteLine("Order is already being processed");
    
    public void Ship(Order order)
    {
        Console.WriteLine("Shipping order...");
        order.SetState(new ShippedOrderState());
    }
    
    public void Cancel(Order order)
    {
        Console.WriteLine("Cannot cancel: Order is being processed");
    }
}

public class ShippedOrderState : IOrderState
{
    public void Process(Order order)
        => Console.WriteLine("Cannot process: Order already shipped");
    
    public void Ship(Order order)
        => Console.WriteLine("Order already shipped");
    
    public void Cancel(Order order)
        => Console.WriteLine("Cannot cancel: Order already shipped");
}

public class Order
{
    private IOrderState _state;
    
    public Order() => _state = new NewOrderState();
    
    public void SetState(IOrderState state) => _state = state;
    
    public void Process() => _state.Process(this);
    public void Ship() => _state.Ship(this);
    public void Cancel() => _state.Cancel(this);
}
```
**Use Case:** Workflows, state machines, finite state transitions

### Design Pattern Selection Framework:

```
┌─────────────────────────────────────────────────────────────┐
│                 When to Use Which Pattern?                  │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Need to create objects?                                    │
│  ├── One instance?              → Singleton                │
│  ├── Family of objects?         → Abstract Factory         │
│  ├── Step-by-step creation?     → Builder                  │
│  └── Based on type?             → Factory Method           │
│                                                             │
│  Need to structure classes?                                 │
│  ├── Incompatible interfaces?   → Adapter                  │
│  ├── Add responsibilities?      → Decorator                │
│  ├── Simplify subsystem?        → Facade                   │
│  └── Control access?            → Proxy                    │
│                                                             │
│  Need object interaction?                                  │
│  ├── Notify changes?            → Observer                 │
│  ├── Interchangeable algorithms?→ Strategy                 │
│  ├── Encapsulate requests?      → Command                  │
│  ├── Processing pipeline?       → Chain of Responsibility  │
│  └── State-dependent behavior?  → State                    │
└─────────────────────────────────────────────────────────────┘
```

**Senior Insight:**
> "Patterns are solutions to common problems, not rules. At my current company:
> 
> - **Repository + UnitOfWork** pattern for data access
> - **Mediator pattern** (MediatR) for CQRS
> - **Specification pattern** for complex queries
> - **Pipeline pattern** for request processing
> 
> We use patterns to:
> 1. **Improve maintainability** (easier to understand)
> 2. **Enable testing** (mocking becomes simpler)
> 3. **Allow extensions** (open/closed principle)
> 4. **Standardize development** (common vocabulary)
> 
> However, we avoid over-engineering with patterns. Sometimes, a simple solution is better than a pattern-driven one."

---

## 14. Dependency Injection

### What is Dependency Injection?

**Definition:** A design pattern where objects receive their dependencies from an external source rather than creating them internally.

### Without DI (Tight Coupling):
```csharp
public class OrderService
{
    private readonly PaymentService _paymentService;  // Hard dependency
    private readonly EmailService _emailService;      // Hard dependency
    private readonly DatabaseContext _context;        // Hard dependency
    
    public OrderService()
    {
        // Creates dependencies internally - HARD TO TEST
        _paymentService = new PaymentService();
        _emailService = new EmailService();
        _context = new DatabaseContext();
    }
}
```

### With DI (Loose Coupling):
```csharp
public class OrderService
{
    private readonly IPaymentService _paymentService;
    private readonly IEmailService _emailService;
    private readonly IRepository<Order> _orderRepository;
    
    public OrderService(
        IPaymentService paymentService,
        IEmailService emailService,
        IRepository<Order> orderRepository)
    {
        // Dependencies injected from outside - EASY TO TEST
        _paymentService = paymentService;
        _emailService = emailService;
        _orderRepository = orderRepository;
    }
}
```

### Types of Dependency Injection:

**1. Constructor Injection** (Most Common)
```csharp
public class ProductController : Controller
{
    private readonly IProductService _productService;
    private readonly ILogger<ProductController> _logger;
    
    public ProductController(IProductService productService, ILogger<ProductController> logger)
    {
        _productService = productService;
        _logger = logger;
    }
}
```

**2. Property Injection** (Optional Dependencies)
```csharp
public class NotificationService
{
    [Inject]
    public IEmailService EmailService { get; set; }
    
    [Inject]
    public ISmsService SmsService { get; set; }
}
```

**3. Method Injection** (Per-Call Dependencies)
```csharp
public class PaymentProcessor
{
    public void Process(IPaymentGateway gateway, PaymentRequest request)
    {
        gateway.Process(request);
    }
}
```

### DI Container Configuration:

```csharp
public class Startup
{
    public void ConfigureServices(IServiceCollection services)
    {
        // 1. Add default services
        services.AddControllers();
        services.AddEndpointsApiExplorer();
        services.AddSwaggerGen();
        
        // 2. Register application services
        services.AddScoped<IProductService, ProductService>();
        services.AddScoped<IOrderService, OrderService>();
        services.AddSingleton<ICacheService, RedisCacheService>();
        services.AddTransient<IEmailService, SmtpEmailService>();
        
        // 3. Register repositories
        services.AddScoped(typeof(IRepository<>), typeof(Repository<>));
        
        // 4. Register HttpClient
        services.AddHttpClient<IPaymentService, PaymentService>(client =>
        {
            client.BaseAddress = new Uri("https://api.payment.com/");
            client.Timeout = TimeSpan.FromSeconds(30);
        });
        
        // 5. Register with factory
        services.AddScoped<IDataService>(provider =>
        {
            var config = provider.GetRequiredService<IConfiguration>();
            var connectionString = config.GetConnectionString("DefaultConnection");
            return new DataService(connectionString);
        });
        
        // 6. Register multiple implementations
        services.AddScoped<IValidator, EmailValidator>();
        services.AddScoped<IValidator, PhoneValidator>();
        services.AddScoped<IValidator, PasswordValidator>();
        
        // 7. Register with specific key
        services.AddKeyedScoped<IAuthService, GoogleAuthService>("google");
        services.AddKeyedScoped<IAuthService, FacebookAuthService>("facebook");
    }
}
```

### Advanced DI Scenarios:

**1. Decorator Pattern with DI:**
```csharp
services.AddScoped<IRepository, Repository>();
services.Decorate<IRepository, CachedRepository>();
services.Decorate<IRepository, LoggingRepository>();
```

**2. Factory Pattern with DI:**
```csharp
public interface IPaymentFactory
{
    IPaymentService Create(string provider);
}

public class PaymentFactory : IPaymentFactory
{
    private readonly IServiceProvider _serviceProvider;
    
    public PaymentFactory(IServiceProvider serviceProvider)
        => _serviceProvider = serviceProvider;
    
    public IPaymentService Create(string provider)
        => provider.ToLower() switch
        {
            "stripe" => _serviceProvider.GetRequiredKeyedService<IPaymentService>("stripe"),
            "paypal" => _serviceProvider.GetRequiredKeyedService<IPaymentService>("paypal"),
            _ => throw new ArgumentException($"Unknown provider: {provider}")
        };
}
```

**3. Scoped Services in Background Services:**
```csharp
public class BackgroundProcessor : BackgroundService
{
    private readonly IServiceProvider _serviceProvider;
    
    public BackgroundProcessor(IServiceProvider serviceProvider)
        => _serviceProvider = serviceProvider;
    
    protected override async Task ExecuteAsync(CancellationToken stoppingToken)
    {
        while (!stoppingToken.IsCancellationRequested)
        {
            using var scope = _serviceProvider.CreateScope();
            var service = scope.ServiceProvider.GetRequiredService<IRepository>();
            
            // Use service...
            
            await Task.Delay(1000, stoppingToken);
        }
    }
}
```

### DI Lifetimes:

| Lifetime | When to Use | Example |
|----------|-------------|---------|
| **Transient** | Stateless, lightweight services | Logging, utility classes |
| **Scoped** | Per-request services | DbContext, repositories |
| **Singleton** | Stateful, shared services | Configuration, caching, HttpClient |

### Testing with DI:

```csharp
public class OrderServiceTests
{
    [Fact]
    public async Task CreateOrder_Should_ProcessPayment()
    {
        // Arrange
        var mockPayment = new Mock<IPaymentService>();
        var mockEmail = new Mock<IEmailService>();
        var mockRepo = new Mock<IRepository<Order>>();
        
        mockPayment.Setup(x => x.ProcessAsync(It.IsAny<PaymentRequest>()))
            .ReturnsAsync(new PaymentResult { Success = true });
        
        var service = new OrderService(
            mockPayment.Object,
            mockEmail.Object,
            mockRepo.Object);
        
        // Act
        var result = await service.CreateOrderAsync(new OrderRequest());
        
        // Assert
        Assert.True(result.Success);
        mockPayment.Verify(x => x.ProcessAsync(It.IsAny<PaymentRequest>()), Times.Once);
    }
}
```

### Best Practices (Senior Level):

1. **Register dependencies at composition root**
   ```csharp
   // ✅ Good
   public class Program
   {
       public static void Main(string[] args)
       {
           var host = CreateHostBuilder(args).Build();
           // Composition root here
           host.Run();
       }
   }
   
   // ❌ Bad
   public class OrderService
   {
       public OrderService()
       {
           // Not here
           var service = new Service();
       }
   }
   ```

2. **Use constructor injection as primary**
   ```csharp
   // ✅ Good
   public class ProductService
   {
       public ProductService(IRepository repository) { }
   }
   ```

3. **Avoid service locator pattern**
   ```csharp
   // ❌ Bad - Service Locator
   public class OrderService
   {
       public void Process()
       {
           var service = ServiceProvider.GetService<IPaymentService>();
       }
   }
   ```

4. **Use named/typed clients for HttpClient**
   ```csharp
   // ✅ Good
   services.AddHttpClient<GithubService>();
   services.AddHttpClient<PaymentService>();
   ```

5. **Validate dependencies at startup**
   ```csharp
   services.AddOptions<EmailSettings>()
       .Validate(s => !string.IsNullOrEmpty(s.SmtpHost))
       .ValidateOnStart();
   ```

**Senior Insight:**
> "In our microservices, we use DI with:
> 
> **1. Autofac** (more advanced than built-in)
> - Property injection for optional dependencies
> - Module-based registration
> - Dynamic proxy for AOP
> 
> **2. Feature Flags with DI:**
> ```csharp
> if (featureManager.IsEnabled("NewPaymentFlow"))
> {
>     services.AddScoped<IPaymentService, NewPaymentService>();
> }
> else
> {
>     services.AddScoped<IPaymentService, LegacyPaymentService>();
> }
> ```
> 
> **3. Lazy Initialization:**
> ```csharp
> public class OrderService
> {
>     private readonly Lazy<IPaymentService> _paymentService;
>     
>     public OrderService(Lazy<IPaymentService> paymentService)
>         => _paymentService = paymentService;
> }
> ```
> 
> **4. Keyed Services (.NET 8):**
> ```csharp
> services.AddKeyedScoped<IPaymentService, StripeService>("stripe");
> services.AddKeyedScoped<IPaymentService, PayPalService>("paypal");
> 
> // Usage
> public class PaymentFactory
> {
>     public IPaymentService Create(string provider)
>         => _serviceProvider.GetRequiredKeyedService<IPaymentService>(provider);
> }
> ```

---

## 15. Async vs Sync Call

### Synchronous vs Asynchronous:

| Aspect | Synchronous | Asynchronous |
|--------|-------------|--------------|
| **Thread Blocking** | ✅ Blocks thread | ❌ Doesn't block |
| **Scalability** | ❌ Poor (thread pool exhaustion) | ✅ Excellent |
| **Responsiveness** | ❌ UI freezes | ✅ UI responsive |
| **Complexity** | ✅ Simple | ❌ More complex |
| **I/O Operations** | ❌ Inefficient | ✅ Efficient |
| **CPU Operations** | ✅ Good | ❌ No benefit |

### Real Example:

```csharp
// ❌ Synchronous (Blocking)
public class OrderController : Controller
{
    private readonly IOrderService _orderService;
    private readonly IEmailService _emailService;
    private readonly IPaymentService _paymentService;
    
    public IActionResult CreateOrder(OrderRequest request)
    {
        // Each call blocks the thread
        var payment = _paymentService.Process(request);     // 2 seconds
        var order = _orderService.Create(request);          // 1 second
        var email = _emailService.SendConfirmation(request); // 1 second
        
        // Total: 4 seconds - THREAD BLOCKED
        return Ok(order);
    }
}

// ✅ Asynchronous (Non-blocking)
public class OrderController : Controller
{
    private readonly IOrderService _orderService;
    private readonly IEmailService _emailService;
    private readonly IPaymentService _paymentService;
    
    public async Task<IActionResult> CreateOrder(OrderRequest request)
    {
        // Each call releases the thread
        var paymentTask = _paymentService.ProcessAsync(request);
        var orderTask = _orderService.CreateAsync(request);
        var emailTask = _emailService.SendConfirmationAsync(request);
        
        // Parallel execution
        await Task.WhenAll(paymentTask, orderTask, emailTask);
        
        // Total: ~2 seconds (max of tasks)
        return Ok(orderTask.Result);
    }
}
```

### Async Patterns:

**1. Fire and Forget (No Await)**
```csharp
// ❌ Bad - Missing await
public async Task ProcessOrder(Order order)
{
    // Fire and forget
    _ = SendEmailAsync(order);  // Exception will be swallowed
}

// ✅ Good - Fire and forget with error handling
public async Task ProcessOrder(Order order)
{
    _ = Task.Run(async () =>
    {
        try
        {
            await SendEmailAsync(order);
        }
        catch (Exception ex)
        {
            _logger.LogError(ex, "Email failed for order {OrderId}", order.Id);
        }
    });
}
```

**2. Parallel Execution**
```csharp
public async Task<Dashboard> GetDashboardAsync()
{
    var tasks = new List<Task>
    {
        GetUserStatsAsync(),
        GetOrderStatsAsync(),
        GetPaymentStatsAsync()
    };
    
    await Task.WhenAll(tasks);
    return new Dashboard
    {
        UserStats = (UserStats)await tasks[0],
        OrderStats = (OrderStats)await tasks[1],
        PaymentStats = (PaymentStats)await tasks[2]
    };
}
```

**3. Cancellation Support**
```csharp
public async Task<IEnumerable<Product>> SearchProductsAsync(
    string query, 
    CancellationToken cancellationToken)
{
    if (string.IsNullOrEmpty(query))
        throw new ArgumentException("Query is required");
    
    return await _productRepository
        .SearchAsync(query, cancellationToken)
        .ToListAsync(cancellationToken);
}
```

**4. Timeout Pattern**
```csharp
public async Task<Order> GetOrderWithTimeoutAsync(int id)
{
    using var cts = new CancellationTokenSource(TimeSpan.FromSeconds(5));
    
    try
    {
        return await GetOrderAsync(id, cts.Token);
    }
    catch (OperationCanceledException)
    {
        _logger.LogWarning("Order retrieval timed out for {OrderId}", id);
        throw new TimeoutException("Order retrieval timed out");
    }
}
```

### Async Best Practices:

```csharp
// 1. Async all the way
public async Task<Order> GetOrderAsync(int id)
    => await _repository.GetAsync(id);  // ✅ Good

// 2. Avoid async void (except event handlers)
public async void Button_Click(object sender, EventArgs e)  // ✅ Exception
    => await ProcessAsync();

public async Task ProcessAsync()  // ✅ Good

// 3. Use ValueTask for frequently cached results
public async ValueTask<Product> GetProductAsync(int id)
{
    if (_cache.TryGetValue(id, out Product product))
        return product;
    
    product = await _repository.GetAsync(id);
    _cache.Set(id, product);
    return product;
}

// 4. Avoid .Result and .Wait()
// ❌ Bad
var result = _service.GetAsync().Result;

// ✅ Good
var result = await _service.GetAsync();

// 5. ConfigureAwait for library code
public async Task<Data> GetDataAsync()
{
    // For library code
    var data = await _repository.GetAsync().ConfigureAwait(false);
    return data;
}
```

### Performance Comparison:

```csharp
[Benchmark]
public async Task<Order> GetOrderSync()
{
    // 3 sequential calls = 3 seconds
    var user = _userService.GetUser();
    var payment = _paymentService.GetPayment();
    var order = _orderService.GetOrder();
    return order;
}

[Benchmark]
public async Task<Order> GetOrderAsync()
{
    // 3 parallel calls = 1 second (if they each take 1 second)
    var userTask = _userService.GetUserAsync();
    var paymentTask = _paymentService.GetPaymentAsync();
    var orderTask = _orderService.GetOrderAsync();
    
    await Task.WhenAll(userTask, paymentTask, orderTask);
    return orderTask.Result;
}
```

**Senior Insight:**
> "We've dramatically improved our system performance by using async/await:
> 
> **Before (Sync):**
> - API Gateway: 200ms average response
> - 10,000 concurrent users → 500 threads blocked
> - Thread pool exhaustion
> 
> **After (Async):**
> - API Gateway: 50ms average response
> - 10,000 concurrent users → 20 active threads
> - 10x throughput improvement
> 
> **Key Lessons:**
> 1. **Async all the way** - Don't mix sync and async
> 2. **Use CancellationToken** - For timeout and graceful shutdown
> 3. **Monitor async tasks** - Use App Insights for tracking
> 4. **Avoid `async void`** - Makes error handling impossible
> 5. **Use `ConfigureAwait(false)`** - In library code for better performance"

---

## 16. Using Spanner over SQL Server - Pros/Cons

### Google Cloud Spanner vs SQL Server Comparison:

| Feature | Spanner | SQL Server |
|---------|---------|------------|
| **Type** | Cloud-native, distributed | Traditional RDBMS |
| **Architecture** | Multi-region, strongly consistent | Single/multi-AZ |
| **Scaling** | ✅ Automatic horizontal | ❌ Vertical (scale-up) / Read replicas |
| **Transactions** | ✅ Global ACID across regions | ✅ ACID (within cluster) |
| **Availability** | 99.999% (5 nines) | 99.99% (with Always On) |
| **Pricing** | 💰 Expensive (pay per use) | 💰💰 Licensing + infrastructure |
| **Schema Evolution** | ✅ Online DDL | ⚠️ Can lock tables |
| **SQL Support** | ✅ Standard SQL | ✅ Extensive T-SQL |
| **Stored Procedures** | ❌ No | ✅ Yes |
| **Triggers** | ❌ No | ✅ Yes |
| **Secondary Indexes** | ✅ Yes | ✅ Yes |
| **Foreign Keys** | ✅ Yes | ✅ Yes |
| **Management** | ✅ Fully managed | ⚠️ Self-managed (or Azure managed) |

### Pros of Spanner:

**1. Global Scale**
```sql
-- Create a globally distributed table
CREATE TABLE Orders (
    OrderId INT64 NOT NULL,
    CustomerId INT64 NOT NULL,
    OrderDate DATE,
    Amount NUMERIC
) PRIMARY KEY (OrderId, CustomerId)
-- Data is automatically partitioned and distributed globally
```

**2. Strong Consistency**
```
┌─────────────┐
│   US-East   │───┐
├─────────────┤   │
│   Europe    │───┼──► TrueTime API (Global Clock)
├─────────────┤   │
│   Asia      │───┘
└─────────────┘

All reads see the latest writes globally (external consistency)
```

**3. Automatic Sharding**
```csharp
// No need to think about sharding - Spanner handles it
public async Task InsertOrderAsync(Order order)
{
    var cmd = connection.CreateCommand();
    cmd.CommandText = @"
        INSERT INTO Orders (OrderId, CustomerId, Amount, OrderDate)
        VALUES (@OrderId, @CustomerId, @Amount, @OrderDate)";
    
    // Spanner automatically distributes data
    await cmd.ExecuteNonQueryAsync();
}
```

**4. Online Schema Migrations**
```sql
-- No downtime for schema changes
ALTER TABLE Orders ADD COLUMN ShippingAddress STRING(MAX);
ALTER TABLE Orders ALTER COLUMN Amount NUMERIC NOT NULL;
-- All without locking the table
```

**5. Point-in-Time Recovery**
```csharp
// Read data as of 30 minutes ago
var connection = new SpannerConnection("...");
var cmd = connection.CreateCommand();
cmd.CommandText = "SELECT * FROM Orders WHERE OrderId = @OrderId";
cmd.Parameters.AddWithValue("@OrderId", orderId);
cmd.TimestampBound = TimestampBound.OfReadTimestamp(
    Timestamp.FromDateTime(DateTime.UtcNow.AddMinutes(-30))
);
```

### Cons of Spanner:

**1. Cost**
```csharp
// Pricing example (2024):
// Instance: $0.90/hour per node (min 3 nodes)
// Storage: $0.30/GB/month
// Network: $0.12/GB outbound

// 3 nodes = $0.90 * 24 * 30 = $648/month (minimum)
// 100GB storage = $30/month
// Total: ~$700+ per month
```

**2. Limited SQL Features**
```sql
-- ❌ Not supported in Spanner
-- Stored procedures
CREATE PROCEDURE UpdateInventory ...
    NOT SUPPORTED

-- Triggers
CREATE TRIGGER ... 
    NOT SUPPORTED

-- Full-text search
CONTAINS(...) 
    NOT SUPPORTED

-- Complex JOINs with subqueries
-- Performance can be unpredictable
```

**3. Connection Pooling**
```csharp
// Need to manage connections differently
// Each connection can be expensive
public class SpannerConnectionManager
{
    // Spanner recommends short-lived connections
    // vs SQL Server where connection pooling is standard
}
```

**4. Query Performance**
```sql
-- ❌ Bad pattern (cross-shard join)
SELECT o.*, c.* 
FROM Orders o 
JOIN Customers c ON o.CustomerId = c.CustomerId
WHERE o.OrderDate > '2024-01-01';

-- ✅ Good pattern (single-shard query)
-- Use interleaved tables
CREATE TABLE Customers (
    CustomerId INT64 NOT NULL,
    Name STRING(MAX)
) PRIMARY KEY (CustomerId);

CREATE TABLE Orders (
    CustomerId INT64 NOT NULL,
    OrderId INT64 NOT NULL,
    Amount NUMERIC
) PRIMARY KEY (CustomerId, OrderId),
INTERLEAVE IN PARENT Customers ON DELETE CASCADE;
```

**5. Migration Complexity**
```csharp
// Moving from SQL Server to Spanner requires:
// 1. Data migration (schema changes)
// 2. Query rewriting (no stored procedures)
// 3. Application changes (no identity columns)
// 4. Transaction behavior changes (stronger consistency)
```

### When to Choose Spanner:

✅ **Choose Spanner when:**
- Global application (users across multiple regions)
- Need strong consistency globally
- Massive scale (100s TB, billions of rows)
- Can afford the cost
- No need for stored procedures/triggers
- Greenfield project

❌ **Don't choose Spanner when:**
- Single region application
- Small to medium scale (< 100GB)
- Heavy use of stored procedures/triggers
- Cost-sensitive
- Legacy SQL Server application
- Need for complex reporting queries

### Hybrid Approach:

```csharp
public class DataAccessService
{
    private readonly SpannerConnection _spanner;
    private readonly SqlServerConnection _sqlServer;
    
    public async Task<Order> GetOrderAsync(int id)
    {
        // Read from Spanner for global data
        return await GetFromSpannerAsync(id);
    }
    
    public async Task UpdateOrderAsync(Order order)
    {
        // Write to Spanner
        await UpdateSpannerAsync(order);
        
        // Sync to SQL Server for reporting
        await SyncToSqlServerAsync(order);
    }
    
    public async Task<SalesReport> GetSalesReportAsync(DateTime date)
    {
        // Complex reports use SQL Server
        return await GetReportFromSqlServerAsync(date);
    }
}
```

### Migration Strategy:

```csharp
public class DatabaseMigration
{
    public async Task MigrateDataAsync()
    {
        // 1. Export from SQL Server
        var data = await ExportSqlServerDataAsync();
        
        // 2. Transform for Spanner
        var spannerData = TransformForSpanner(data);
        
        // 3. Import to Spanner (batch)
        await BulkInsertToSpannerAsync(spannerData);
        
        // 4. Verify consistency
        await VerifyDataConsistencyAsync();
        
        // 5. Cutover (blue-green)
        await SwitchTrafficToSpannerAsync();
    }
}
```

**Senior Insight:**
> "We evaluated Spanner for our global e-commerce platform:
> 
> **Why we considered it:**
> - Users across 3 continents
> - Need for consistency in order processing
> - 100M+ transactions per day
> 
> **Why we chose a hybrid approach:**
> - Cost: Spanner would cost ~$100k/month vs $20k/month for SQL Server
> - Existing T-SQL heavy applications
> - Complex reporting requirements
> 
> **Our Solution:**
> - **Spanner**: Global customer profiles (read-heavy)
> - **SQL Server**: Order processing (write-heavy, ACID)
> - **Elasticsearch**: Search and analytics
> - **Redis**: Caching layer
> 
> **The Magic:**
> - Service Mesh (Istio) for smart routing
> - CQRS: Write to SQL Server, read from cache/Spanner
> - Event-driven sync between databases
> 
> **Takeaway:**
> Spanner is powerful but expensive. Use it only if you truly need global scale with strong consistency. For most enterprises, SQL Server (especially Azure SQL Managed Instance) provides sufficient scale at lower cost."

---

## 🎯 Final Interview Tips (9+ Years Experience)

### 1. **Demonstrate Leadership**
- Talk about mentoring junior developers
- Architecture decisions you've made
- Code review practices you've implemented

### 2. **Show Business Impact**
- "Reduced response time by 60%"
- "Saved $50k/year in cloud costs"
- "Increased team productivity by 40%"

### 3. **Discuss Trade-offs**
- "We chose Entity Framework for productivity but used Dapper for reporting"
- "Microservices gave us autonomy but increased operational complexity"

### 4. **Technical Depth**
- Memory management (GC, IDisposable)
- Performance optimization (Profiling, Benchmarking)
- Security (OWASP, Authentication, Encryption)

### 5. **Modern .NET Knowledge**
- .NET 8 features (primary constructors, collection expressions)
- Minimal APIs vs Controllers
- gRPC vs REST
- Blazor vs React/Angular

### 6. **Cloud & DevOps**
- Containerization (Docker, Kubernetes)
- CI/CD (GitHub Actions, Azure DevOps)
- Infrastructure as Code (Terraform, ARM)

### 7. **System Design**
- Event-driven architecture
- Saga pattern
- CQRS
- Event Sourcing

### 8. **Soft Skills**
- Communicate complex concepts clearly
- Show you can work with stakeholders
- Discuss conflict resolution examples

---

**Good luck with your interview!** With 9+ years of experience, you have the depth to handle any question. The key is to show you understand the **"why"** behind patterns and technologies, not just the **"how"**.